# Silver: Sector Assignment

Assigns jobs to industry sectors using deterministic rules and fallback logic.

## Assignment Strategy

1. **Company-based** (deterministic): Match company to known sector mapping
2. **Title-based** (pattern matching): Use job title keywords
3. **Skill-based** (aggregation): Infer from extracted skills
4. **ML-based** (future): Predict sector from description
5. **Default**: Queue for manual review if confidence < threshold

## Sectors

- Technology, Finance, Healthcare, Retail, Manufacturing, Education, etc.
- Extendable taxonomy in `metadata.sector_dim`

In [0]:
dbutils.widgets.text("confidence_threshold", "0.7", "Minimum confidence for auto-assignment")
dbutils.widgets.dropdown("queue_low_confidence", "true", ["true", "false"], "Queue low confidence for review")

confidence_threshold = float(dbutils.widgets.get("confidence_threshold"))
queue_low_confidence = dbutils.widgets.get("queue_low_confidence") == "true"

print(f"Confidence Threshold: {confidence_threshold}")
print(f"Queue Low Confidence: {queue_low_confidence}")

In [0]:
import json
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import *

CATALOG = "workspace"
SILVER_SCHEMA = f"{CATALOG}.silver"
METADATA_SCHEMA = f"{CATALOG}.metadata"

run_timestamp = F.current_timestamp()
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

# Sector assignment rules (company name patterns)
COMPANY_SECTOR_PATTERNS = {
    "technology": ["software", "tech", "data", "cloud", "ai", "digital", "cyber", "saas"],
    "finance": ["bank", "financial", "fintech", "insurance", "investment", "capital"],
    "healthcare": ["health", "medical", "pharma", "biotech", "hospital", "clinic"],
    "retail": ["retail", "ecommerce", "shop", "store", "marketplace"],
    "manufacturing": ["manufacturing", "automotive", "industrial", "engineering"],
    "education": ["education", "university", "school", "learning", "training"],
    "consulting": ["consulting", "advisory", "strategy"],
    "media": ["media", "publishing", "entertainment", "streaming"],
    "energy": ["energy", "oil", "gas", "renewable", "utility"],
    "transportation": ["logistics", "transport", "shipping", "delivery"]
}

# Title-based sector keywords
TITLE_SECTOR_KEYWORDS = {
    "technology": ["developer", "engineer", "data scientist", "devops", "software", "programmer", "architect"],
    "finance": ["accountant", "financial analyst", "auditor", "controller", "trader"],
    "healthcare": ["nurse", "doctor", "clinician", "therapist", "medical"],
    "education": ["teacher", "professor", "instructor", "tutor"],
    "consulting": ["consultant", "advisor", "analyst"],
    "marketing": ["marketing", "brand manager", "seo", "content"],
    "sales": ["sales", "account executive", "business development"]
}

print(f"Loaded {len(COMPANY_SECTOR_PATTERNS)} sector categories")

In [0]:
# Load jobs without sector assignment
jobs_df = spark.sql(f"""
SELECT 
    enterprise_job_id,
    source_name,
    company_name_norm,
    title_normalized,
    description_raw
FROM {SILVER_SCHEMA}.silver_jobs_current
WHERE is_active = true 
  AND soft_delete_flag = false
""")

job_count = jobs_df.count()
print(f"Jobs to assign sectors: {job_count}")

if job_count == 0:
    dbutils.notebook.exit({"status": "success", "message": "No jobs to process"})

In [0]:
def assign_sector_by_company(company_name):
    """Assign sector based on company name patterns"""
    if not company_name:
        return None, 0.0
    
    company_lower = company_name.lower()
    
    for sector, patterns in COMPANY_SECTOR_PATTERNS.items():
        for pattern in patterns:
            if pattern in company_lower:
                return sector, 0.85  # High confidence for company match
    
    return None, 0.0

def assign_sector_by_title(title):
    """Assign sector based on job title keywords"""
    if not title:
        return None, 0.0
    
    title_lower = title.lower()
    
    for sector, keywords in TITLE_SECTOR_KEYWORDS.items():
        for keyword in keywords:
            if keyword in title_lower:
                return sector, 0.75  # Moderate confidence for title match
    
    return None, 0.0

def assign_sector_combined(company, title):
    """Combined sector assignment with fallback"""
    # Try company first (higher confidence)
    sector_company, conf_company = assign_sector_by_company(company)
    if sector_company and conf_company >= 0.70:
        return (sector_company, "company_pattern", conf_company)
    
    # Fallback to title
    sector_title, conf_title = assign_sector_by_title(title)
    if sector_title:
        return (sector_title, "title_keyword", conf_title)
    
    # Default: unknown
    return ("unknown", "no_match", 0.0)

# Register UDF
sector_assign_udf = F.udf(assign_sector_combined, StructType([
    StructField("sector", StringType()),
    StructField("assignment_method", StringType()),
    StructField("confidence", DoubleType())
]))

print("Sector assignment functions registered")

In [0]:
# Assign sectors
sector_assigned_df = jobs_df.withColumn(
    "sector_assignment",
    sector_assign_udf(F.col("company_name_norm"), F.col("title_normalized"))
).select(
    "*",
    F.col("sector_assignment.sector").alias("assigned_sector"),
    F.col("sector_assignment.assignment_method").alias("assignment_method"),
    F.col("sector_assignment.confidence").alias("assignment_confidence")
).drop("sector_assignment")

# Add metadata
sector_assigned_df = sector_assigned_df.withColumn(
    "assigned_at", run_timestamp
).withColumn(
    "batch_id", F.lit(run_id)
)

print("Sector assignment complete")
display(sector_assigned_df.limit(10))

In [0]:
if queue_low_confidence:
    low_confidence_df = sector_assigned_df.where(
        F.col("assignment_confidence") < confidence_threshold
    )
    
    low_conf_count = low_confidence_df.count()
    
    if low_conf_count > 0:
        # Create review queue table
        spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.silver_semantic_review_queue (
          review_id STRING,
          enterprise_job_id STRING,
          issue_type STRING,
          issue_detail STRING,
          confidence DECIMAL(5,4),
          queued_at TIMESTAMP,
          status STRING,
          resolved_at TIMESTAMP,
          resolution_notes STRING
        )
        USING DELTA
        """)
        
        # Queue for review
        review_queue_df = low_confidence_df.select(
            F.expr("uuid()").alias("review_id"),
            F.col("enterprise_job_id"),
            F.lit("SECTOR_LOW_CONFIDENCE").alias("issue_type"),
            F.concat(
                F.lit("Sector assignment confidence below threshold: "),
                F.col("assigned_sector"),
                F.lit(" ("),
                F.col("assignment_method"),
                F.lit(")")
            ).alias("issue_detail"),
            F.col("assignment_confidence").cast("decimal(5,4)").alias("confidence"),
            run_timestamp.alias("queued_at"),
            F.lit("PENDING").alias("status")
        )
        
        review_queue_df.write.format("delta").mode("append").saveAsTable(
            f"{SILVER_SCHEMA}.silver_semantic_review_queue"
        )
        
        print(f"Queued {low_conf_count} jobs for manual review")
    else:
        low_conf_count = 0
else:
    low_conf_count = 0

In [0]:
# Update current table with sector assignments
from delta.tables import DeltaTable

# Check if sector columns exist, if not add them
try:
    current_schema = spark.table(f"{SILVER_SCHEMA}.silver_jobs_current").schema
    current_columns = [field.name for field in current_schema.fields]
    
    missing_columns = []
    if "sector_assigned" not in current_columns:
        missing_columns.append("ADD COLUMN sector_assigned STRING")
    if "sector_assignment_method" not in current_columns:
        missing_columns.append("ADD COLUMN sector_assignment_method STRING")
    if "sector_confidence" not in current_columns:
        missing_columns.append("ADD COLUMN sector_confidence DOUBLE")
    if "sector_assigned_at" not in current_columns:
        missing_columns.append("ADD COLUMN sector_assigned_at TIMESTAMP")
    
    if missing_columns:
        for col_def in missing_columns:
            spark.sql(f"ALTER TABLE {SILVER_SCHEMA}.silver_jobs_current {col_def}")
        print(f"Added {len(missing_columns)} sector tracking columns to current table")
    
    sector_update_df = sector_assigned_df.select(
        "enterprise_job_id",
        "assigned_sector",
        "assignment_method",
        "assignment_confidence",
        "assigned_at"
    )
    
    delta_current = DeltaTable.forName(spark, f"{SILVER_SCHEMA}.silver_jobs_current")
    
    delta_current.alias("cur").merge(
        sector_update_df.alias("sec"),
        "cur.enterprise_job_id = sec.enterprise_job_id"
    ).whenMatchedUpdate(
        set={
            "sector_assigned": "sec.assigned_sector",
            "sector_assignment_method": "sec.assignment_method",
            "sector_confidence": "sec.assignment_confidence",
            "sector_assigned_at": "sec.assigned_at"
        }
    ).execute()
    
    print("Sector assignments updated in current table")
except Exception as e:
    print(f"Warning: Could not update sectors: {e}")

In [0]:
# Sector distribution
sector_summary_df = sector_assigned_df.groupBy("assigned_sector").agg(
    F.count("*").alias("job_count"),
    F.avg("assignment_confidence").alias("avg_confidence")
).orderBy(F.desc("job_count"))

print("\n=== Sector Distribution ===")
display(sector_summary_df)

# Assignment method breakdown
method_summary_df = sector_assigned_df.groupBy("assignment_method").count().orderBy(F.desc("count"))
print("\n=== Assignment Methods ===")
display(method_summary_df)

# Exit summary
result = {
    "status": "success",
    "run_id": run_id,
    "jobs_processed": job_count,
    "high_confidence_count": sector_assigned_df.where(F.col("assignment_confidence") >= confidence_threshold).count(),
    "low_confidence_count": low_conf_count,
    "unknown_sector_count": sector_assigned_df.where(F.col("assigned_sector") == "unknown").count()
}

print("\n=== Sector Assignment Summary ===")
print(json.dumps(result, indent=2))

dbutils.notebook.exit(json.dumps(result))